In [1]:
# NOTEBOOK 43 - dashboard export for the FINAL model (dyn_adv): playback, wind edges, summary and the what-if scenarios on the same 970 test windows
# Notebook 10 made these files for v2. This writes the same schema from the final model (5 seeds, real day-old forecast wind) into a NEW folder, so nothing existing changes:
#   v2_clean/dashboard_export_dynadv/   stations.json (copy) | playback.json | wind_edges.json | summary.json | whatif.json
# whatif.json holds 13 scenarios computed on every one of the 970 windows (live mode in notebook 41 adds 3 weather scenarios, which cannot be rebuilt for historical windows):
#   wind (forecast wind only): x0.25 / x0.5 / x1.5 / x2, NW inflow, reversed | graph: wind graph off / fixed distance graph / direction flipped
#   emission: the 5 worst hotspots (highest mean PM2.5 in the test period) cut by 30%, every station cut by 20% | outage: worst hotspot offline, the 5 worst offline
# Each scenario also gets a REAL check on all 970 windows: the mean change and the share of windows that move the expected way. This is the validation of the scenarios.
# Checks printed: window indices equal the old playback, forecasts equal notebook 39's saved predictions, actuals equal the old playback, and the feature rebuild used for the
# emission / outage scenarios reproduces the original features exactly when nothing is changed.
# Time: about 10 minutes on the GPU. A progress line prints per scenario. LIMIT = a small number tests it quickly.
# HOW TO CHECK: this cell only prints a note.
LIMIT = None
KEEP = None     # None = all 13 scenarios; or a list of keys, e.g. ['wind_x0_5', 'wind_x2', 'cut_hotspots_30'], to shrink whatif.json (about 3.7 MB per scenario for all windows)
print('dashboard export for the final model; LIMIT =', LIMIT, '| KEEP =', KEEP)

dashboard export for the final model; LIMIT = None | KEEP = None


In [2]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
PREVDAY_NPY     = os.path.join(V2_DIR, 'openmeteo_prevday1_uv.npy')
ADJ_FC_NPY      = os.path.join(V2_DIR, 'dynamic_adj_prevday1.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

Device cuda | F=26 | n_fut=10 | seeds [42, 7, 123, 2024, 999]


In [3]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])
dyn_adj = np.load(DYN_ADJ_OM_NPY).astype(np.float32)                 # A(t) from ERA5 wind (T,N,N)
static_adj = np.load(STATIC_ADJ_NPY).astype(np.float32)
np.fill_diagonal(static_adj, 0.0)
assert dyn_adj.shape == (T, N, N)
print(f'T={T}  dyn_adj(ERA5) {dyn_adj.shape}  static edges {int((static_adj>0).sum())}')
prevday = np.load(PREVDAY_NPY).astype(np.float32)                    # day-old forecast wind (T,N,2), blowing TOWARD
adj_fc = np.load(ADJ_FC_NPY).astype(np.float32)                      # forecast-wind graph for the future hours (T,N,N)
assert prevday.shape == era5.shape and adj_fc.shape == dyn_adj.shape, (prevday.shape, adj_fc.shape)
print('forecast wind', prevday.shape, '| forecast graph', adj_fc.shape, '| mean |forecast - ERA5| wind speed difference (m/s):',
      float(np.abs(np.hypot(prevday[..., 0], prevday[..., 1]) - era5_spd).mean()))

T=19921  dyn_adj(ERA5) (19921, 37, 37)  static edges 508
forecast wind (19921, 37, 2) | forecast graph (19921, 37, 37) | mean |forecast - ERA5| wind speed difference (m/s): 0.7865347862243652


In [4]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

CPCB_COLS = (BASE_FEATS +
    ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
Xc = np.zeros((T, N, len(CPCB_COLS)), dtype=np.float32)
for fi, col in enumerate(CPCB_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xc[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

built = (BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + CPCB_COLS[len(BASE_FEATS):])
assert built == FEATURE_COLS, 'feature order mismatch vs v3_config'
X_raw = np.concatenate([
    Xc[:, :, :len(BASE_FEATS)],
    era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None],
    Xc[:, :, len(BASE_FEATS):],
], axis=2).astype(np.float32)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')
FUT_IDX   = TIME_IDX + [U_IDX, V_IDX, W_IDX]
print(f'F={X_raw.shape[2]}  NaNs {int(np.isnan(X_raw).sum())}')

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_28884\1524811802.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month


F=26  NaNs 0


In [5]:
# 1) geometry and the two new feature pairs for Delhi
lats = df_st['latitude'].values.astype(float); lons = df_st['longitude'].values.astype(float)
THRESHOLD_KM, MAX_LAG = 12.0, 6

def geometry(lats, lons):
    n = len(lats)
    dist = np.zeros((n, n)); du = np.zeros((n, n)); dv = np.zeros((n, n))
    R = 6371.0
    for i in range(n):
        for j in range(n):
            if i != j:
                p1, p2 = np.radians(lats[i]), np.radians(lats[j])
                a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lons[j] - lons[i]) / 2) ** 2
                dist[i, j] = 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
                de = (lons[j] - lons[i]) * np.cos(np.radians(lats[i])); dn = lats[j] - lats[i]; nr = np.hypot(de, dn)
                if nr > 0:
                    du[i, j], dv[i, j] = de / nr, dn / nr          # unit vector from i to j
    return dist, du, dv, (dist > 0) & (dist <= THRESHOLD_KM)

def transport_features(pm, uv, dist, du, dv, within):
    # pm (T,n) raw PM2.5, uv (T,n,2) ERA5 wind (blowing TOWARD). Only values at or before hour t are used (no leakage).
    T, n = pm.shape
    adv = pm.copy(); nb = pm.copy()
    src = np.arange(n)[:, None] * np.ones((1, n), dtype=int)           # source station index j for entry [j, i]
    stat = within.astype(float); cnt = stat.sum(0); has_nb = cnt > 0
    for t in range(T):
        align = uv[t, :, 0][:, None] * du + uv[t, :, 1][:, None] * dv   # wind at j along the direction j -> i  (m/s)
        w = np.where(within & (align > 0), align / (dist + 1e-6), 0.0)  # j is upwind of i
        wsum = w.sum(0); has = wsum > 0
        lag = np.clip(np.rint(dist / (np.maximum(align, 0.5) * 3.6)), 0, MAX_LAG).astype(int)   # hours of travel (km / (km/h))
        vals = pm[np.clip(t - lag, 0, T - 1), src]                      # pm[t - lag[j,i], j]
        adv[t, has] = ((w * vals).sum(0))[has] / wsum[has]
        nb[t, has_nb] = ((stat * pm[t][:, None]).sum(0))[has_nb] / cnt[has_nb]
    return adv, nb

dist, du, dv, within = geometry(lats, lons)
t0 = time.time()
adv, nb = transport_features(X_raw[:, :, 0].astype(float), era5, dist, du, dv, within)
pm0 = X_raw[:, :, 0]
extra = np.stack([adv - 0, adv - pm0, nb - 0, nb - pm0], axis=2).astype(np.float32)       # adv_pm, adv_diff, nb_pm, nb_diff
extra[:, :, 0] = adv; extra[:, :, 2] = nb
X30 = np.concatenate([X_raw, extra], axis=2)
print(f'features built in {time.time()-t0:.0f}s | X30 {X30.shape}')
print('mean |adv - own| :', float(np.abs(adv - pm0).mean()), '| mean |nb - own| :', float(np.abs(nb - pm0).mean()))
print('corr(adv, own PM):', float(np.corrcoef(adv.ravel(), pm0.ravel())[0, 1]), '| corr(nb, own PM):', float(np.corrcoef(nb.ravel(), pm0.ravel())[0, 1]))

# HOW TO CHECK: about a minute. Both correlations are high (0.8-0.95) because neighbouring stations resemble each other;
# the question for the models is whether the lagged, wind-aligned version adds anything beyond the same-hour mean.

features built in 1s | X30 (19921, 37, 30)
mean |adv - own| : 28.104649143919474 | mean |nb - own| : 23.36611597159713
corr(adv, own PM): 0.8577609401778713 | corr(nb, own PM): 0.9041709851392609


In [6]:
# 2) scaler for the 30 channels (fit on the training period only); the first 26 must match the saved v3 scaler
from sklearn.preprocessing import StandardScaler
import pickle
T = X30.shape[0]; tr_end = int(T * 0.70)
sc30 = StandardScaler().fit(X30[:tr_end].reshape(-1, 30))
v3s = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
assert np.allclose(sc30.mean_[:26], v3s.mean_, rtol=1e-4, atol=1e-4) and np.allclose(sc30.scale_[:26], v3s.scale_, rtol=1e-4, atol=1e-4), 'scaler mismatch'
log_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Xs_full = sc30.transform(X30.reshape(-1, 30)).reshape(T, N, 30).astype(np.float32)
Yl_full = to_log(X30[:, :, 0])
Y_raw = X30[:, :, 0].copy()
print('scaler check passed: the first 26 channels match the v3 scaler')

# HOW TO CHECK: prints the pass line. An assertion error means the data or train split changed - stop and tell me.

scaler check passed: the first 26 channels match the v3 scaler


C:\Users\jlakk_j7eh5fd\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# future-hour wind channels from the day-old forecast (scaled with the same train-period scaler as the observed wind)
pv_spd = np.hypot(prevday[..., 0], prevday[..., 1])
pv_raw = np.stack([prevday[..., 0], prevday[..., 1], pv_spd], axis=2).astype(np.float32)          # (T,N,3)
WIDX = [U_IDX, V_IDX, W_IDX]
Xs_fut = Xs_full.copy()
Xs_fut[:, :, WIDX] = ((pv_raw - sc30.mean_[WIDX]) / sc30.scale_[WIDX]).astype(np.float32)
print('future wind channels now come from the day-old forecast | scaled means: observed', Xs_full[:, :, WIDX].mean((0, 1)).round(2), 'forecast', Xs_fut[:, :, WIDX].mean((0, 1)).round(2))

# HOW TO CHECK: the u and v means are close; the forecast speed mean is lower than the observed one (about -0.5 against -0.1 on the scaled axis) because the day-old forecast runs slower than ERA5. That bias is real and is why the models are retrained on it.

future wind channels now come from the day-old forecast | scaled means: observed [ 0.04  0.01 -0.13] forecast [-0.07  0.14 -0.5 ]


In [8]:
# model with three graph modes plus the learned blend, and optional context dropout at training time
CTX_COLS = ['city_mean_pm', 'city_max_pm', 'pm_minus_city']
CTX_IDX = [FEATURE_COLS.index(c) for c in CTX_COLS]
WSPD_IDX = FEATURE_COLS.index('wspd')
FUT_WSPD = 9                                  # wspd is the last of the 10 future channels (7 calendar + u, v, wspd)

class WATGCNv4(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon, n_stations,
                 dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        sa = sa.clone(); sa.fill_diagonal_(0)
        self.register_buffer('static_adj', sa, persistent=False)
        self.register_buffer('static_rn', sa / sa.sum(1, keepdim=True).clamp(min=1.0), persistent=False)   # row-normalised
        if graph_mode == 'blend':
            self.blend_a = nn.Parameter(torch.zeros(1)); self.blend_b = nn.Parameter(torch.zeros(1))

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def alpha(self, speed):                       # speed: (B, steps) standardised wind speed -> weight of the wind graph
        return torch.sigmoid(self.blend_a + self.blend_b * speed)[..., None, None]

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            A = None
        elif self.graph_mode == 'dynamic':
            A = self._n(At + I.view(1, 1, n, n))
        elif self.graph_mode == 'static':
            A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
        else:                                      # blend
            a = self.alpha(X[..., WSPD_IDX].mean(-1))
            A = self._n(a * At + (1 - a) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n))
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(h if A is None else torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        elif self.graph_mode == 'static':
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        else:
            af = self.alpha(Fut[..., FUT_WSPD].mean(-1))
            ctx_fc = torch.matmul(self._n(af * Afc + (1 - af) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n)), ctx_rep)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, self.fut_enc(Fut)], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

In [9]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def sc(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))

In [10]:
# the 970 playback windows, the 5 final models, and a generic dataset so any scenario can swap the inputs, the graphs or the future wind
SEEDS = [42, 7, 123, 2024, 999]
ADV_COLS = list(range(26)) + [26, 27]
STATIC_NP = np.load(STATIC_ADJ_NPY).astype(np.float32)
val_end = int(T * 0.85); n_test = (T - val_end) - (INPUT_WINDOW + HORIZON) + 1
assert n_test == 2918, n_test
TS = pd.DatetimeIndex(timestamps)
N_PLAYBACK = 970
sidx = np.linspace(0, n_test - 1, N_PLAYBACK).astype(int)
OLD_DIR = os.path.join(V2_DIR, 'dashboard_export')
old_pb = json.load(open(os.path.join(OLD_DIR, 'playback.json')))
assert [int(f['window_idx']) for f in old_pb] == sidx.tolist(), 'the playback window set changed'
if LIMIT: sidx = sidx[:LIMIT]; old_pb = old_pb[:LIMIT]
starts = val_end + sidx
EXPORT_DIR = os.path.join(V2_DIR, 'dashboard_export_dynadv'); os.makedirs(EXPORT_DIR, exist_ok=True)

class DSX(Dataset):
    def __init__(self, starts, Xin, At, Xfut, Afc, Yl): self.s, self.Xin, self.At, self.Xfut, self.Afc, self.Yl = np.asarray(starts, np.int64), Xin, At, Xfut, Afc, Yl
    def __len__(self): return len(self.s)
    def __getitem__(self, k):
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(np.ascontiguousarray(self.Xin[g:i1])), torch.from_numpy(self.At[g:i1].copy()), torch.from_numpy(self.Afc[i1:i2].copy()),
                torch.from_numpy(np.ascontiguousarray(self.Xfut[i1:i2])), torch.from_numpy(self.Yl[i1 - 1].copy()), torch.from_numpy(self.Yl[i1:i2].copy()), torch.tensor(1.0))

models = []
for sd in SEEDS:
    m = WATGCNv4(len(ADV_COLS), HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, N_FUT, GCN_HOPS, graph_mode='dynamic', static_adj=STATIC_NP).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v5dyn_adv_seed{sd}.pt'), map_location=DEVICE)); m.eval(); models.append(m)

def forecast(Xin, At, Xfut, Afc, Yl):
    ld = DataLoader(DSX(starts, Xin, At, Xfut, Afc, Yl), batch_size=32, shuffle=False, num_workers=0)
    outs = [predict(m, ld)[0] for m in models]
    return from_log(np.mean(outs, axis=0))                                        # (windows, 24, N) ug/m3, ensemble mean in log space

# baseline inputs (the forecast setting of notebook 39)
XIN0 = np.ascontiguousarray(Xs_full[:, :, ADV_COLS])
FUT0 = np.ascontiguousarray(Xs_fut[:, :, FUT_IDX])
P0 = forecast(XIN0, dyn_adj, FUT0, adj_fc, Yl_full)
saved = np.load(os.path.join(RESULTS_DIR, 'dynadv_test_pred_forecast.npy'))[sidx]
actual = np.stack([Y_raw[g + INPUT_WINDOW: g + INPUT_WINDOW + HORIZON] for g in starts])                    # (windows, 24, N)
raw_last = np.stack([Y_raw[g + INPUT_WINDOW - 1] for g in starts])
print(f'windows {len(starts)} | baseline mean forecast {P0.mean():.1f} ug/m3')
print(f'check A: forecasts vs notebook 39 saved predictions, max abs diff {np.abs(P0 - saved).max():.4f}   (expect about 0.001)')
old_act = np.array([[[st['actual_24h'][h] for st in f['stations']] for h in range(HORIZON)] for f in old_pb], dtype=float)
print(f'check B: actuals vs the old playback.json, mean abs diff {np.abs(actual - old_act).mean():.3f}   (expect below 0.1)')

# HOW TO CHECK: the two checks print values as described. If A is above 0.05 or B above 1, stop and tell me.

windows 970 | baseline mean forecast 128.5 ug/m3
check A: forecasts vs notebook 39 saved predictions, max abs diff 0.0108   (expect about 0.001)
check B: actuals vs the old playback.json, mean abs diff 0.009   (expect below 0.1)


In [11]:
# helpers for the scenarios: forecast graph from a wind field, the future-channel array, and the feature rebuild for emission / outage scenarios
def adj_from_wind(uv, block=1500):
    n_t = uv.shape[0]; out = np.zeros((n_t, N, N), dtype=np.float32)
    for s in range(0, n_t, block):
        u, v = uv[s:s + block, :, 0].astype(np.float64), uv[s:s + block, :, 1].astype(np.float64)
        align = u[:, :, None] * du[None] + v[:, :, None] * dv[None]
        A = np.where(within[None] & (align > 0), align / (dist[None] + 1e-6), 0.0).astype(np.float32)
        rs = A.sum(2, keepdims=True); rs[rs == 0] = 1
        out[s:s + block] = A / rs
    return out

CAL_IDX = FUT_IDX[:7]; WIND3 = [U_IDX, V_IDX, W_IDX]
def fut_array(uv):                                                                  # (T,N,10): 7 calendar channels + the scaled forecast wind (u, v, speed)
    raw3 = np.stack([uv[..., 0], uv[..., 1], np.hypot(uv[..., 0], uv[..., 1])], axis=2).astype(np.float32)
    return np.concatenate([Xs_full[:, :, CAL_IDX], ((raw3 - sc30.mean_[WIND3]) / sc30.scale_[WIND3]).astype(np.float32)], axis=2)
assert np.abs(fut_array(prevday) - FUT0).max() < 1e-5, 'future-channel recipe differs from the baseline'
chk = adj_from_wind(prevday[:3000]); print('graph recipe check (forecast graph, first 3000 hours), max abs diff vs stored:', float(np.abs(chk - adj_fc[:3000]).max())); del chk

BASE8 = X_raw[:, :, :8].copy()                                                       # pm2.5, pm10, no2, so2, co, o3, rh, at
def rebuild(base_mod):
    Xr = X_raw.copy(); Xr[:, :, :8] = base_mod
    pm = pd.DataFrame(base_mod[:, :, 0].astype(np.float64))
    d = {'pm_roll6_mean': pm.rolling(6, min_periods=1).mean().values, 'pm_roll6_std': pm.rolling(6, min_periods=1).std().fillna(0.0).values,
         'pm_roll24_mean': pm.rolling(24, min_periods=1).mean().values, 'pm_roll24_std': pm.rolling(24, min_periods=1).std().fillna(0.0).values,
         'pm_delta3': (pm - pm.shift(3)).fillna(0.0).values}
    cm = pm.mean(axis=1).values[:, None]
    d['city_mean_pm'] = np.broadcast_to(cm, pm.shape); d['city_max_pm'] = np.broadcast_to(pm.max(axis=1).values[:, None], pm.shape); d['pm_minus_city'] = pm.values - cm
    for k, v in d.items(): Xr[:, :, FEATURE_COLS.index(k)] = v
    adv, _ = transport_features(Xr[:, :, 0].astype(float), era5, dist, du, dv, within)
    X28 = np.concatenate([Xr, adv[:, :, None].astype(np.float32), (adv - Xr[:, :, 0])[:, :, None].astype(np.float32)], axis=2)
    return ((X28 - sc30.mean_[:28]) / sc30.scale_[:28]).astype(np.float32), to_log(Xr[:, :, 0])
t0 = time.time(); Xs28, Yl28 = rebuild(BASE8)
print(f'check C: feature rebuild with nothing changed vs the original features: max abs diff {np.abs(Xs28 - XIN0).max():.5f} (scaled units) and {np.abs(Yl28 - Yl_full).max():.5f} (target), {time.time() - t0:.0f}s   (expect below 0.001)')
del Xs28, Yl28

# HOW TO CHECK: check C is below 0.001. If it is larger, the emission and outage scenarios would not be comparable to the baseline - stop and tell me.

graph recipe check (forecast graph, first 3000 hours), max abs diff vs stored: 0.0
check C: feature rebuild with nothing changed vs the original features: max abs diff 0.00000 (scaled units) and 0.00000 (target), 1s   (expect below 0.001)


In [12]:
# the 13 scenarios on all windows
hot = [int(i) for i in np.argsort(-Y_raw[val_end:].mean(axis=0))[:5]]
print('5 chronic hotspots (highest mean PM2.5 in the test period):', [station_names[i] for i in hot])
def wind_uv(kind):
    uv = prevday.copy()
    if kind.startswith('x'): uv = uv * float(kind[1:].replace('_', '.'))
    elif kind == 'nw': sp = np.hypot(uv[..., 0], uv[..., 1]); uv = np.stack([sp * 0.7071, -sp * 0.7071], -1).astype(np.float32)
    elif kind == 'rev': uv = -uv
    return uv.astype(np.float32)
def offline_arrays(ids):
    b = BASE8.copy(); off = np.zeros(N, bool); off[ids] = True
    for i in ids:
        nbrs = within[i] & ~off
        for c in range(6):                                                           # pollutants only; the station reports nothing, neighbours fill in (the training imputation, tier 1)
            b[:, i, c] = b[:, nbrs, c].mean(axis=1) if nbrs.any() else b[:, ~off, c].mean(axis=1)
    return b
def cut_arrays(ids, factor):
    b = BASE8.copy(); b[:, ids, 0] = b[:, ids, 0] * factor; return b

SCEN = [('wind_x0_25', 'Calm: wind x0.25', 'wind', 'pos'), ('wind_x0_5', 'Wind halved', 'wind', 'pos'), ('wind_x1_5', 'Wind x1.5', 'wind', 'neg'), ('wind_x2', 'Wind doubled', 'wind', 'neg'),
        ('wind_nw', 'NW inflow (stubble smoke)', 'wind', 'small'), ('wind_rev', 'Wind reversed', 'wind', 'small'),
        ('graph_off', 'Wind graph switched off', 'graph', None), ('graph_static', 'Fixed distance graph instead of the wind graph', 'graph', None), ('graph_flip', 'Wind graph direction flipped', 'graph', None),
        ('cut_hotspots_30', 'Cut the 5 worst hotspots by 30%', 'emission', 'neg'), ('cut_city_20', 'Cut every station by 20%', 'emission', 'neg'),
        ('offline_top1', 'Worst hotspot sensor offline', 'outage', 'small'), ('offline_top5', '5 worst hotspot sensors offline', 'outage', 'small')]
def run_scenario(key):
    if key.startswith('wind_'):
        uv = wind_uv({'wind_x0_25': 'x0_25', 'wind_x0_5': 'x0_5', 'wind_x1_5': 'x1_5', 'wind_x2': 'x2', 'wind_nw': 'nw', 'wind_rev': 'rev'}[key])
        afc = adj_from_wind(uv); P = forecast(XIN0, dyn_adj, fut_array(uv), afc, Yl_full); del afc; return P
    if key == 'graph_off': z = np.zeros_like(dyn_adj); return forecast(XIN0, z, FUT0, z, Yl_full)
    if key == 'graph_static': s = np.broadcast_to(STATIC_NP, dyn_adj.shape); return forecast(XIN0, s, FUT0, s, Yl_full)
    if key == 'graph_flip': return forecast(XIN0, dyn_adj.transpose(0, 2, 1).copy(), FUT0, adj_fc.transpose(0, 2, 1).copy(), Yl_full)
    b = {'cut_hotspots_30': lambda: cut_arrays(hot, 0.7), 'cut_city_20': lambda: cut_arrays(list(range(N)), 0.8),
         'offline_top1': lambda: offline_arrays(hot[:1]), 'offline_top5': lambda: offline_arrays(hot)}[key]()
    Xin, Yl = rebuild(b); return forecast(np.ascontiguousarray(Xin[:, :, :28]), dyn_adj, FUT0, adj_fc, Yl)

base_w = P0.mean(axis=(1, 2)); RESULTS, scen_meta = {}, []
t0 = time.time()
for key, label, group, expected in SCEN:
    if KEEP and key not in KEEP: continue
    P = run_scenario(key); d_w = P.mean(axis=(1, 2)) - base_w                       # per-window change of the mean forecast
    if expected is None: check, share = 'exploratory', None
    else:
        share = float(((d_w < 0) if expected == 'neg' else (d_w > 0) if expected == 'pos' else (np.abs(d_w) <= 0.05 * base_w)).mean())
        check = 'as expected' if share >= 0.9 else 'unexpected'
    scen_meta.append({'key': key, 'label': label, 'group': group, 'mean_change': round(float(d_w.mean()), 1), 'share_windows_as_expected': None if share is None else round(share, 3), 'check': check})
    RESULTS[key] = P
    print(f'  {key:16} done | mean change {d_w.mean():+7.1f} ug/m3 | windows as expected {"n/a" if share is None else f"{100 * share:.0f}%"} | {check} | {time.time() - t0:.0f}s')

# HOW TO CHECK: one line per scenario. Wind speed x2 / x1.5 negative, x0.5 / x0.25 positive, direction near zero, cuts negative, outages near zero; "windows as expected" should be 90%+ for those.

5 chronic hotspots (highest mean PM2.5 in the test period): ['Anand Vihar', 'Jahangirpuri', 'Rohini', 'Wazirpur', 'Nehru Nagar']
  wind_x0_25       done | mean change   +17.8 ug/m3 | windows as expected 100% | as expected | 3s
  wind_x0_5        done | mean change   +11.8 ug/m3 | windows as expected 100% | as expected | 6s
  wind_x1_5        done | mean change   -11.0 ug/m3 | windows as expected 100% | as expected | 10s
  wind_x2          done | mean change   -20.8 ug/m3 | windows as expected 100% | as expected | 13s
  wind_nw          done | mean change    -1.1 ug/m3 | windows as expected 90% | as expected | 16s
  wind_rev         done | mean change    +8.5 ug/m3 | windows as expected 42% | unexpected | 19s
  graph_off        done | mean change    +0.7 ug/m3 | windows as expected n/a | exploratory | 22s
  graph_static     done | mean change    -0.7 ug/m3 | windows as expected n/a | exploratory | 24s
  graph_flip       done | mean change    +0.6 ug/m3 | windows as expected n/a | explor

In [13]:
# write the five files (same schema as notebook 10 plus the new fields on each scenario)
def aqi_category(v): return 'Good' if v <= 30 else 'Satisfactory' if v <= 60 else 'Moderate' if v <= 90 else 'Poor' if v <= 120 else 'Very Poor' if v <= 250 else 'Severe'
CPCB_THRESHOLD, WHO_THRESHOLD = 60.0, 15.0
import shutil
shutil.copy(os.path.join(OLD_DIR, 'stations.json'), os.path.join(EXPORT_DIR, 'stations.json'))

now_ts = TS[val_end + INPUT_WINDOW: val_end + INPUT_WINDOW + n_test]
playback = []
for b, idx in enumerate(sidx):
    e = {'window_idx': int(idx), 'timestamp': pd.Timestamp(now_ts[idx]).strftime('%Y-%m-%d %H:%M'), 'stations': []}
    for s in range(N):
        a24 = [round(float(x), 1) for x in actual[b, :, s]]; p24 = [round(float(x), 1) for x in P0[b, :, s]]; cur = round(float(raw_last[b, s]), 1)
        e['stations'].append({'station_id': s, 'current_pm25': cur, 'current_aqi': aqi_category(cur), 'actual_24h': a24, 'predicted_24h': p24, 'peak_predicted': round(max(p24), 1),
                              'hours_to_breach_cpcb': next((h + 1 for h, v in enumerate(p24) if v > CPCB_THRESHOLD), None)})
    playback.append(e)
assert [f['timestamp'] for f in playback] == [f['timestamp'] for f in old_pb], 'timestamps differ from the old playback'
json.dump(playback, open(os.path.join(EXPORT_DIR, 'playback.json'), 'w'))

we = {}
EDGE_MIN, EDGE_CAP = 0.03, 120
for b, idx in enumerate(sidx):
    A = dyn_adj[starts[b] + INPUT_WINDOW - 1]; ii, jj = np.where(A > EDGE_MIN); ww = A[ii, jj]; keep = np.argsort(ww)[::-1][:EDGE_CAP]
    we[str(int(idx))] = [[int(ii[k]), int(jj[k]), round(float(ww[k]), 3)] for k in keep]
json.dump(we, open(os.path.join(EXPORT_DIR, 'wind_edges.json'), 'w'), separators=(',', ':'))

fm = json.load(open(os.path.join(RESULTS_DIR, 'dynadv_final_metrics.json')))
summary = {'model_name': 'WA-TGCN dyn_adv (5-seed ensemble, real day-old forecast wind)', 'n_stations': N, 'test_windows_total': int(n_test), 'test_windows_exported': len(playback),
           'test_period': f"{pd.Timestamp(now_ts[0]).strftime('%Y-%m-%d')} to {pd.Timestamp(now_ts[-1]).strftime('%Y-%m-%d')}",
           'rmse': round(fm['overall']['forecast']['test']['rmse'], 2), 'mae': round(fm['overall']['forecast']['test']['mae'], 2), 'r2': round(fm['overall']['forecast']['test']['r2'], 4),
           'rmse_by_horizon': {k: round(v, 2) for k, v in fm['by_horizon']['forecast'].items()}, 'rmse_by_regime': {k: round(v, 2) for k, v in fm['by_volatility']['forecast'].items()},
           'persistence_rmse': round(fm['persistence']['rmse'], 2), 'rmse_perfect_wind': round(fm['overall']['perfect']['test']['rmse'], 2),
           'cpcb_threshold': CPCB_THRESHOLD, 'who_threshold': WHO_THRESHOLD}
json.dump(summary, open(os.path.join(EXPORT_DIR, 'summary.json'), 'w'), indent=2)

wf = {'scenarios': scen_meta, 'cpcb_threshold': CPCB_THRESHOLD, 'windows': {}}
for b, idx in enumerate(sidx):
    wf['windows'][str(int(idx))] = {}
    for key in RESULTS:
        P = RESULTS[key][b]                                                              # (24, N)
        wf['windows'][str(int(idx))][key] = {'predicted_24h': np.round(P).astype(int).T.tolist(), 'peak': np.round(P.max(axis=0)).astype(int).tolist(),
                                              'breach_h': [int(next((h + 1 for h, v_ in enumerate(P[:, i]) if v_ > CPCB_THRESHOLD), 0)) for i in range(N)]}
json.dump(wf, open(os.path.join(EXPORT_DIR, 'whatif.json'), 'w'), separators=(',', ':'))

print(f"{'file':16}{'size':>10}")
for f in ('stations.json', 'playback.json', 'wind_edges.json', 'summary.json', 'whatif.json'): print(f'{f:16}{os.path.getsize(os.path.join(EXPORT_DIR, f)) / 1e6:>8.1f} MB')
print(f"\nsummary: RMSE {summary['rmse']} | MAE {summary['mae']} | R2 {summary['r2']} | perfect-wind RMSE {summary['rmse_perfect_wind']} | {summary['test_period']}")
print(f'windows {len(playback)} | edges per window {np.mean([len(v) for v in we.values()]):.0f}')
print('written to', EXPORT_DIR)

# HOW TO CHECK: the old and new playback timestamps matched (no assertion error above), summary RMSE is 58.32. whatif.json is the large file: if it is much more than 40 MB, tell me and we trim scenarios.

file                  size
stations.json        0.0 MB
playback.json       17.0 MB
wind_edges.json      1.6 MB
summary.json         0.0 MB
whatif.json         44.2 MB

summary: RMSE 58.32 | MAE 38.21 | R2 0.6647 | perfect-wind RMSE 57.71 | 2025-12-08 to 2026-04-09
windows 970 | edges per window 120
written to D:\FYP\Eco-Twin\v2_clean\dashboard_export_dynadv
